# core

> Factory helpers that build a ready-to-train agent and heading arena.


In [ ]:
#| default_exp core


In [ ]:
#| hide
from nbdev.showdoc import *


In [ ]:
#| export
import numpy as np
from explRL.params import Params
from explRL.basis import vonMisesBasis
from explRL.agents import flexiblePolicy, PolicyGradientRL, flexibleAgent
from explRL.envs import HeadingArena
from explRL.prestructured import prestructuredPolicy, prestructuredAgent, GoalHeadingRL
from explRL.circuit import sinGoal
from explRL.circuit_agent import circuitPolicy, circuitAgent, CircuitRL


In [ ]:
#| export
def loadAgent(params=None, goal=None, seed=None):
    'Build a flexiblePolicy, PolicyGradientRL, and flexibleAgent at a random heading.'
    if params is None:
        params = Params()
    if goal is None:
        goal = np.pi
    rng = np.random.default_rng(seed)
    fk = vonMisesBasis(params.nb, params.nx, params.kappa)
    nb = params.nb
    omega = np.concatenate([np.full(nb, params.omegaF0), np.full(nb, params.omegaS0)])
    policy = flexiblePolicy(omega=omega, fk=fk, wParams=params.wParams)
    rl = PolicyGradientRL(policy, lr=params.lr)
    theta = 2 * np.pi * rng.integers(params.nx) / params.nx
    return flexibleAgent(theta, goal, policy, rl, params, rng=rng)

def loadEnv(params=None, goal=None):
    'Build a HeadingArena; goal defaults to pi radians.'
    if goal is None:
        goal = np.pi
    return HeadingArena(goal=goal)

def loadPrestructuredAgent(params=None, goal=None, seed=None, thetaG0=None, learn=True):
    'Build a prestructuredPolicy, GoalHeadingRL, and prestructuredAgent.'
    if params is None:
        params = Params()
    if goal is None:
        goal = np.pi
    rng = np.random.default_rng(seed)
    thetaA0 = float(rng.uniform(0, 2 * np.pi))
    g0 = float(rng.uniform(0, 2 * np.pi)) if thetaG0 is None else float(thetaG0)
    policy = prestructuredPolicy(g0)
    rl = GoalHeadingRL(policy, lr=params.lrG if learn else 0.0)
    return prestructuredAgent(thetaA0, goal, policy, rl, params, rng=rng)

def loadCircuitAgent(params=None, seed=None, wG0=None, wC0=None, learnC=True, learnG=True, jumps=True):
    'Build a circuitPolicy, CircuitRL, and circuitAgent on the heading grid.'
    if params is None:
        params = Params()
    N = params.nCirc
    rng = np.random.default_rng(seed)
    a = int(rng.integers(N))
    wC = rng.uniform(0, 1, (N, N)) if wC0 is None else np.asarray(wC0, dtype=float)
    if wG0 is None:
        wG = sinGoal(float(rng.uniform(0.05, 0.5)), float(rng.uniform(0, 2 * np.pi)), N)
    else:
        wG = np.asarray(wG0, dtype=float)
    policy = circuitPolicy(wG, wC)
    rl = CircuitRL(policy, learnC=learnC, learnG=learnG, jumps=jumps)
    return circuitAgent(a, policy, rl, params, rng=rng)


In [ ]:
from math import pi
from fastcore.test import test_eq, test_close

p = Params()
agent = loadAgent(params=p, seed=0)
test_eq(agent.policy.omega.size, 2 * p.nb)
test_close(agent.policy.omega[:p.nb], p.omegaF0)
test_close(agent.policy.omega[p.nb:], p.omegaS0)
test_eq(agent.policy.fk.shape, (p.nb, p.nx))
test_eq(agent.rl.lr, p.lr)


In [ ]:
a0 = loadAgent(seed=42)
a1 = loadAgent(seed=42)
test_eq(a0.theta, a1.theta)
headings = {2 * pi * k / p.nx for k in range(p.nx)}
assert a0.theta in headings


In [ ]:
test_close(loadEnv().goal, pi)
test_close(loadAgent().goal, pi)


In [ ]:
from explRL.core import loadPrestructuredAgent
from explRL.prestructured import prestructuredPolicy

a = loadPrestructuredAgent(seed=3)
test_eq(type(a.policy), prestructuredPolicy)
test_close(a.thetaC, a.theta)
test_eq(a.rl.lr, Params().lrG)

b = loadPrestructuredAgent(seed=3, thetaG0=1.2, learn=False)
test_eq(b.policy.thetaG, b.policy.thetaG % (2 * pi))
test_close(b.policy.thetaG, 1.2)
test_eq(b.rl.lr, 0.0)

c0 = loadPrestructuredAgent(seed=11)
c1 = loadPrestructuredAgent(seed=11)
test_eq(c0.theta, c1.theta)
test_close(c0.policy.thetaG, c1.policy.thetaG)
for _ in range(5):
    test_eq(c0.samplePolicy(), c1.samplePolicy())


In [ ]:
from explRL.core import loadCircuitAgent
from explRL.circuit_agent import circuitPolicy

ca = loadCircuitAgent(seed=4)
test_eq(type(ca.policy), circuitPolicy)
test_eq(ca.c, ca.a)
cb0 = loadCircuitAgent(seed=4)
cb1 = loadCircuitAgent(seed=4)
test_eq(cb0.a, cb1.a)
test_close(cb0.policy.wG, cb1.policy.wG)
test_close(cb0.policy.wC, cb1.policy.wC)
test_eq(cb0.sampleSaccade().direction, cb1.sampleSaccade().direction)


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()
